# R5 — Independent replication on NetFlow v2 (NF-UNSW-NB15-v2 → NF-CSE-CIC-IDS2018-v2), binary task. POST HOC.
Run F0 → F5 in order on an A100 runtime. The protocol is written to disk in F0 **before** any model is trained or evaluated.

### CELL F0 — setup, frozen protocol, data, group-disjoint split

In [ ]:
# CELL F0 — setup, frozen protocol, data, group-disjoint split
from google.colab import drive
drive.mount("/content/drive")
import os, gc, json, time, hashlib, shutil
import numpy as np, pandas as pd
BASE = "/content/drive/MyDrive/TrustShift_IDS"
RAW = f"{BASE}/data/raw/NetFlow_v2"
SRC_CSV = f"{RAW}/NF-UNSW-NB15-v2/NF-UNSW-NB15-v2.csv"
EXT_CSV = f"{RAW}/NF-CSE-CIC-IDS2018-v2/data/NF-CSE-CIC-IDS2018-v2.csv"
OUT = f"{BASE}/V2_adaptive_reliability/revision_R5_netflow"; os.makedirs(OUT, exist_ok=True)

DROP = ["IPV4_SRC_ADDR", "IPV4_DST_ADDR", "L4_SRC_PORT", "L4_DST_PORT", "DNS_QUERY_ID"]
cols = pd.read_csv(SRC_CSV, nrows=2).columns.tolist()
assert cols == pd.read_csv(EXT_CSV, nrows=2).columns.tolist(), "source and external schemas differ"
FEATURES = [c for c in cols if c not in DROP + ["Label", "Attack"]]
assert len(FEATURES) == 38
PROTOCOL = {
  "created": time.strftime("%Y-%m-%d %H:%M:%S"), "status": "frozen before any training or evaluation (post hoc replication)",
  "source": "NF-UNSW-NB15-v2", "external": "NF-CSE-CIC-IDS2018-v2", "task": "binary (Label)", "features": FEATURES, "dropped": DROP,
  "split": {"grouping": "identical 38-feature vector (sha1)", "conflicting_label_groups": "removed", "fractions": [0.60, 0.15, 0.10, 0.15], "seed": 42},
  "window": 5000, "risk_budget": 0.01, "coverage_grid": [0.999, 0.995, 0.99, 0.98, 0.975, 0.95, 0.925, 0.90, 0.85, 0.80, 0.75, 0.70, 0.60, 0.50],
  "state_thresholds": {"elevated": 0.99, "severe": 0.90}, "fixed": "loosest threshold with calibration selective risk <= 1%",
  "detectors": {"XGBoost": "hist, depth 8, lr 0.1, up to 1000 trees, early stopping 50 on validation logloss, seed 42",
                "MLP": "log1p + standardise; 256-128-64 ReLU, dropout 0.2, Adam 1e-3, batch 4096, early stopping 5 on validation loss, seed 42"},
  "calibration": "scalar temperature scaling on the calibration partition",
  "monitors": {"ATC": "Garg et al. 2022, leave-window-out boundaries q95/q99 on calibration windows",
               "confidence-only": "window mean risk score, boundaries q95/q99 on calibration windows"},
  "sequences": {"orderings": [1, 7, 21, 84], "stages": ["clean", "mask05", "mask10", "mask20", "combined10", "clean_recovery"], "windows_per_stage": 10,
                "random_mask": "k = round(p*38) features per flow set to the training median", "important_mask": "top-k XGBoost gain features set to the median in all flows"},
  "unseen": {"families": ["Exploits", "Fuzzers", "Generic", "Reconnaissance", "DoS"], "detector": "XGBoost, retrained without the family"},
}
json.dump(PROTOCOL, open(f"{OUT}/F0_PROTOCOL_FROZEN.json", "w"), indent=2)
print("[F0] protocol frozen:", f"{OUT}/F0_PROTOCOL_FROZEN.json")

df = pd.read_csv(SRC_CSV, usecols=FEATURES + ["Label", "Attack"])
X_all = df[FEATURES].to_numpy(np.float64); y_all = df["Label"].to_numpy(np.int64); fam_all = df["Attack"].astype(str).to_numpy()
del df; gc.collect()
h = pd.util.hash_pandas_object(pd.DataFrame(X_all), index=False).to_numpy()
g = pd.DataFrame({"h": h, "y": y_all})
mixed = g.groupby("h").y.nunique(); mixed_h = set(mixed[mixed > 1].index)
keep = ~g.h.isin(mixed_h).to_numpy()
print(f"[F0] flows {len(y_all):,} | groups {g.h.nunique():,} | conflicting-label groups {len(mixed_h):,} ({(~keep).sum():,} flows removed)")
X_all, y_all, fam_all, h = X_all[keep], y_all[keep], fam_all[keep], h[keep]
uh = np.unique(h); rng = np.random.default_rng(42); perm = rng.permutation(len(uh))
cut = np.cumsum([int(f * len(uh)) for f in PROTOCOL["split"]["fractions"]])[:-1]
part_of_group = np.empty(len(uh), np.int8)
for i, idx in enumerate(np.split(perm, cut)): part_of_group[idx] = i
part = part_of_group[np.searchsorted(uh, h)]
NAMES = ["train", "validation", "calibration", "test"]
S = {n: np.where(part == i)[0] for i, n in enumerate(NAMES)}
for n in NAMES: print(f"   {n:11s} {len(S[n]):9,} flows | attack share {y_all[S[n]].mean():.4f}")
assert len(set(h[S['train']]) & set(h[S['test']])) == 0
MEDIAN = np.median(X_all[S["train"]], axis=0)
print("[F0] COMPLETE")

### CELL F1 — train XGBoost and MLP, temperature scaling, thresholds, monitors, source-test results

In [ ]:
# CELL F1 — train XGBoost and MLP, temperature scaling, thresholds, monitors, source-test results
import xgboost as xgb, tensorflow as tf, keras
from scipy.optimize import minimize_scalar
tf.keras.utils.set_random_seed(42)
EPS, W = PROTOCOL["risk_budget"], PROTOCOL["window"]
Xtr, ytr = X_all[S["train"]], y_all[S["train"]]; Xva, yva = X_all[S["validation"]], y_all[S["validation"]]
Xca, yca = X_all[S["calibration"]], y_all[S["calibration"]]; Xte, yte = X_all[S["test"]], y_all[S["test"]]

def train_xgb(Xa, ya, Xv, yv):
    m = xgb.XGBClassifier(tree_method="hist", device="cuda", max_depth=8, learning_rate=0.1, n_estimators=1000,
                          early_stopping_rounds=50, eval_metric="logloss", random_state=42)
    m.fit(Xa, ya, eval_set=[(Xv, yv)], verbose=False); return m
LOGCOLS = None
def mlp_prep_fit(Xa):
    Z = np.log1p(np.clip(Xa, 0, None)); return Z.mean(0), Z.std(0) + 1e-9
def mlp_prep(X, mu, sd): return ((np.log1p(np.clip(X, 0, None)) - mu) / sd).astype(np.float32)

xgb_m = train_xgb(Xtr, ytr, Xva, yva)
mu, sd = mlp_prep_fit(Xtr)
mlp = keras.Sequential([keras.Input((38,)), keras.layers.Dense(256, activation="relu"), keras.layers.Dropout(0.2),
                        keras.layers.Dense(128, activation="relu"), keras.layers.Dropout(0.2), keras.layers.Dense(64, activation="relu"),
                        keras.layers.Dense(1, activation="sigmoid")])
mlp.compile(optimizer=keras.optimizers.Adam(1e-3), loss="binary_crossentropy")
mlp.fit(mlp_prep(Xtr, mu, sd), ytr, validation_data=(mlp_prep(Xva, mu, sd), yva), batch_size=4096, epochs=100, verbose=0,
        callbacks=[keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)])
xgb_m.save_model(f"{OUT}/F1_xgb_source.json"); mlp.save(f"{OUT}/F1_mlp_source.keras"); np.savez(f"{OUT}/F1_mlp_prep.npz", mu=mu, sd=sd, median=MEDIAN)

def p_attack(det, X, bs=65536):
    if det == "XGBoost": return xgb_m.predict_proba(X.astype(np.float32))[:, 1].astype(np.float64)
    return mlp.predict(mlp_prep(X, mu, sd), batch_size=bs, verbose=0).reshape(-1).astype(np.float64)
def logit(p): p = np.clip(p, 1e-12, 1 - 1e-12); return np.log(p) - np.log1p(-p)
def risk_pred(p, T):
    q = 1 / (1 + np.exp(-logit(p) / T)); return 1 - np.maximum(q, 1 - q), (q >= 0.5).astype(np.int64)
def thresholds_from(r, err):
    o = np.argsort(r, kind="mergesort"); ok = np.where(np.cumsum(err[o]) / np.arange(1, len(r) + 1) <= EPS)[0]
    rs = np.sort(r); t = {"fixed": float(r[o][ok.max()]) if len(ok) else float(rs[0])}
    t["elevated"] = float(rs[int(np.ceil(0.99 * len(r))) - 1]); t["severe"] = float(rs[int(np.ceil(0.90 * len(r))) - 1])
    for c in PROTOCOL["coverage_grid"]: t[f"cov_{c:.3f}"] = float(rs[int(np.ceil(c * len(r))) - 1])
    return t
CAL = {}
for det in ["XGBoost", "MLP"]:
    p = p_attack(det, Xca)
    nll = lambda lt: float(np.mean(-(yca * np.log(np.clip(1/(1+np.exp(-logit(p)/np.exp(lt))), 1e-15, 1)) + (1 - yca) * np.log(np.clip(1 - 1/(1+np.exp(-logit(p)/np.exp(lt))), 1e-15, 1)))))
    T = float(np.exp(minimize_scalar(nll, bounds=(-3, 3), method="bounded").x))
    r, pr = risk_pred(p, T); err = pr != yca; conf = 1 - r
    nw = len(r) // W; pw = np.random.default_rng(42).permutation(len(r))[:nw * W].reshape(nw, W)
    atc_null, mean_null = [], []
    for w in range(nw):
        idx = pw[w]; rest = np.ones(len(r), bool); rest[idx] = False
        atc_null.append(float(np.mean(conf[idx] < np.quantile(conf[rest], float(err[rest].mean()))))); mean_null.append(float(r[idx].mean()))
    rt, prt = risk_pred(p_attack(det, Xte), T); et = prt != yte
    CAL[det] = {"T": T, "thresholds": thresholds_from(r, err), "atc_conf_threshold": float(np.quantile(conf, float(err.mean()))),
                "atc_bounds": [float(np.quantile(atc_null, .95)), float(np.quantile(atc_null, .99))],
                "conf_bounds": [float(np.quantile(mean_null, .95)), float(np.quantile(mean_null, .99))], "null_windows": nw,
                "cal_error_rate": float(err.mean()), "test_accuracy": float(1 - et.mean()), "test_attack_recall": float(np.mean(prt[yte == 1] == 1)),
                "test_errors": int(et.sum())}
    print(f"[F1] {det}: T={T:.4f} | cal error {err.mean():.5f} | test acc {CAL[det]['test_accuracy']:.5f} recall {CAL[det]['test_attack_recall']:.4f} | fixed tau {CAL[det]['thresholds']['fixed']:.4g} | null windows {nw}")
imp = xgb_m.get_booster().get_score(importance_type="gain")
IMPORTANT = [FEATURES[int(k[1:])] if k.startswith("f") and k[1:].isdigit() else k for k, _ in sorted(imp.items(), key=lambda kv: -kv[1])]
CAL["importance_order"] = IMPORTANT
json.dump(CAL, open(f"{OUT}/F1_calibration_thresholds.json", "w"), indent=2); print("[F1] top features:", IMPORTANT[:8]); print("[F1] COMPLETE")

### CELL F2 — deployment sequences on the source test partition

In [ ]:
# CELL F2 — deployment sequences on the source test partition
def records(r, pred, y, base, det):
    conf = 1 - r; att = y == 1; err = pred != y; rows = []
    atc = float(np.mean(conf < CAL[det]["atc_conf_threshold"])); mr = float(r.mean())
    for tn, t in CAL[det]["thresholds"].items():
        acc = r <= t
        rows.append({**base, "detector": det, "thr_name": tn, "atc_est": atc, "mean_risk": mr, "n": len(y), "acc_n": int(acc.sum()),
                     "acc_err": int((acc & err).sum()), "n_attack": int(att.sum()), "acc_attack": int((acc & att).sum()),
                     "acc_missed_attack": int((acc & att & (pred == 0)).sum()), "acc_benign": int((acc & ~att).sum()),
                     "acc_false_alarm": int((acc & ~att & (pred == 1)).sum())})
    return rows
fidx = {f: i for i, f in enumerate(FEATURES)}
def mask(X, stage, rng):
    X = X.copy(); n = X.shape[0]
    def rnd(p):
        k = int(round(p * 38)); cols = np.argsort(rng.random((n, 38)), axis=1)[:, :k]; X[np.arange(n)[:, None], cols] = MEDIAN[cols]
    def imp(p):
        k = int(round(p * 38)); cols = [fidx[f] for f in IMPORTANT[:k]]; X[:, cols] = MEDIAN[cols]
    if stage == "mask05": rnd(0.05)
    elif stage == "mask10": rnd(0.10)
    elif stage == "mask20": rnd(0.20)
    elif stage == "combined10": rnd(0.10); imp(0.10)
    return X
stages = PROTOCOL["sequences"]["stages"]; need = len(stages) * 10 * W
assert len(Xte) >= need, f"test partition too small ({len(Xte)} < {need})"
rows = []
for seed in PROTOCOL["sequences"]["orderings"]:
    rng = np.random.default_rng(seed); order = rng.permutation(len(Xte))[:need]
    for si, st in enumerate(stages):
        idx = order[si * 10 * W:(si + 1) * 10 * W]; Xs = mask(Xte[idx], st, rng); ys = yte[idx]
        for det in ["XGBoost", "MLP"]:
            r, pr = risk_pred(p_attack(det, Xs), CAL[det]["T"])
            for w in range(10):
                a, b = w * W, (w + 1) * W
                rows += records(r[a:b], pr[a:b], ys[a:b], {"seed": seed, "stage": st, "window_id": si * 10 + w}, det)
    print(f"  ordering {seed}: done")
pd.DataFrame(rows).to_csv(f"{OUT}/F2_sequence_window_outcomes.csv.gz", index=False); print("[F2] COMPLETE", len(rows))

### CELL F3 — cross-dataset transfer to NF-CSE-CIC-IDS2018-v2 (file order, chunked)

In [ ]:
# CELL F3 — cross-dataset transfer to NF-CSE-CIC-IDS2018-v2 (file order, chunked)
rows = []; carry = None; wid = 0; tot = {d: {"n": 0, "err": 0, "att": 0, "det": 0} for d in ["XGBoost", "MLP"]}
for ch in pd.read_csv(EXT_CSV, usecols=FEATURES + ["Label"], chunksize=1_000_000):
    Xc = ch[FEATURES].to_numpy(np.float64); yc = ch["Label"].to_numpy(np.int64)
    if carry is not None: Xc = np.vstack([carry[0], Xc]); yc = np.concatenate([carry[1], yc])
    nfull = (len(yc) // W) * W; carry = (Xc[nfull:], yc[nfull:]); Xc, yc = Xc[:nfull], yc[:nfull]
    for det in ["XGBoost", "MLP"]:
        r, pr = risk_pred(p_attack(det, Xc), CAL[det]["T"])
        tot[det]["n"] += len(yc); tot[det]["err"] += int((pr != yc).sum()); tot[det]["att"] += int(yc.sum()); tot[det]["det"] += int(((pr == 1) & (yc == 1)).sum())
        for w in range(nfull // W):
            a, b = w * W, (w + 1) * W
            rows += records(r[a:b], pr[a:b], yc[a:b], {"window_id": wid + w}, det)
    wid += nfull // W; print(f"  windows so far: {wid:,}")
for det, t in tot.items():
    print(f"[F3] {det}: flows {t['n']:,} | accuracy {1 - t['err']/t['n']:.4f} | attack share {t['att']/t['n']:.4f} | attack recall {t['det']/t['att']:.4f}")
pd.DataFrame(rows).to_csv(f"{OUT}/F3_external_window_outcomes.csv.gz", index=False); print("[F3] COMPLETE", len(rows), "| tail flows not windowed:", len(carry[1]))

### CELL F4 — unseen attack families (leave-one-attack-out, XGBoost)

In [ ]:
# CELL F4 — unseen attack families (leave-one-attack-out, XGBoost)
rows = []
fam_tr, fam_va, fam_ca, fam_te = (fam_all[S[n]] for n in NAMES)
for fam in PROTOCOL["unseen"]["families"]:
    ktr, kva, kca = fam_tr != fam, fam_va != fam, fam_ca != fam
    m = train_xgb(Xtr[ktr], ytr[ktr], Xva[kva], yva[kva])
    p = m.predict_proba(Xca[kca].astype(np.float32))[:, 1]
    nll = lambda lt: float(np.mean(-(yca[kca] * np.log(np.clip(1/(1+np.exp(-logit(p)/np.exp(lt))), 1e-15, 1)) + (1 - yca[kca]) * np.log(np.clip(1 - 1/(1+np.exp(-logit(p)/np.exp(lt))), 1e-15, 1)))))
    T = float(np.exp(minimize_scalar(nll, bounds=(-3, 3), method="bounded").x))
    held = fam_te == fam; r, pr = risk_pred(m.predict_proba(Xte[held].astype(np.float32))[:, 1], T); miss = pr == 0
    rec = {"family": fam, "held_out_flows": int(held.sum()), "recall_full": float(1 - miss.mean()), "mean_confidence": float((1 - r).mean()), "T": T}
    o = np.argsort(r, kind="mergesort")
    for c in [1.0, 0.995, 0.99, 0.975, 0.95, 0.90, 0.75, 0.50]:
        k = max(1, int(np.ceil(c * len(r)))); rec[f"risk_at_{c:.3f}"] = float(miss[o[:k]].mean())
    rows.append(rec); print(f"[F4] {fam}: n={held.sum():,} recall {rec['recall_full']:.3f} conf {rec['mean_confidence']:.4f} risk@100% {rec['risk_at_1.000']:.3f} risk@50% {rec['risk_at_0.500']:.3f}")
pd.DataFrame(rows).to_csv(f"{OUT}/F4_unseen_family_results.csv", index=False); print("[F4] COMPLETE")

### CELL F5 — package (zip written outside the folder being zipped)

In [ ]:
# CELL F5 — package (zip written outside the folder being zipped)
for f in os.listdir(OUT):
    if f.endswith(".zip"): os.remove(os.path.join(OUT, f))
z = shutil.make_archive(f"{BASE}/V2_adaptive_reliability/TrustShift_revision_R5_export", "zip", OUT)
print("Upload:", z, f"({os.path.getsize(z)/1e6:.1f} MB)"); print("Contents:", sorted(os.listdir(OUT)))